# 6교시 · 통계 분석 기초
### — 내가 본 차이가 진짜인지 재 보기

지금까지 우리는 계속 **숫자를 비교**했습니다.
West 반품률이 높다, 할인하면 이익이 줄어 보인다, 고객 유형별로 주문 금액이 비슷해 보인다.

이번 시간에는 이렇게 묻습니다. **그 차이, 진짜입니까?**

**이 시간이 끝나면 할 수 있는 것**

1. 두 집단의 평균 차이가 우연인지 아닌지 검정할 수 있다
2. p값을 잘못 읽지 않는다
3. 비율끼리 비교할 때는 카이제곱을 쓴다는 걸 안다
4. 상관계수를 계산하고, **그것이 인과가 아니라는 점**을 설명할 수 있다
5. 업무 질문을 "무엇과 무엇을 비교할 것인가"로 바꿀 수 있다

> ### 미리 말씀드립니다
> 이번 시간에 **수식은 하나도 나오지 않습니다.**
> 계산은 파이썬이 합니다. 우리가 할 일은 **어떤 상황에 무엇을 쓰는지 고르고, 나온 숫자를 제대로 읽는 것**입니다.

---
# 6-1. 먼저 예상을 적습니다

**아직 아무 데이터도 보지 마세요.**

지금부터 몇 가지 질문을 드립니다. 지금까지 5교시 동안 본 것과, 여러분이 회사에서 겪은 경험을 바탕으로
**답을 미리 적어 주세요.**

## 왜 미리 적으라고 하는가

결과를 본 다음에는 사람이 거의 항상 **"그럴 줄 알았다"** 고 생각합니다.
빗나갔다는 걸 스스로 알아채지 못합니다. 그래서 데이터를 봐도 배우는 게 없습니다.

**미리 적어 두면 빗나간 것이 눈에 보입니다.** 그게 오늘 배울 것의 대부분입니다.

---

### 아래 셀을 더블클릭해서 채우세요

**1. 할인을 한 주문과 안 한 주문 중 어느 쪽이 이익이 많을 것 같습니까? 차이가 클까요, 작을까요?**

　→ 내 예상:

**2. 고객 유형(Consumer / Corporate)에 따라 주문당 이익이 다를 것 같습니까?**

　→ 내 예상:

**3. 3교시에서 West 지역 반품률이 11.56%로 나왔습니다. 다른 지역은 3% 안팎이었고요.**
**　 이 차이가 "진짜 차이"라고 보십니까, 아니면 "어쩌다 그렇게 나온 것"이라고 보십니까?**

　→ 내 예상:

**4. 할인율과 이익은 얼마나 강하게 연결돼 있을 것 같습니까? (아주 강함 / 어느 정도 / 약함 / 거의 없음)**

　→ 내 예상:

**5. 배송이 예상보다 늦으면 고객 별점이 떨어질 겁니다. 그렇다면 배송을 빨리 하면 별점이 오를까요?**

　→ 내 예상:

---

> **적으셨습니까?** 적으셨다면 다음으로 넘어갑니다.
> **이 셀은 6-12에서 다시 꺼내 봅니다.**

---
## 준비 — 데이터와 도구 불러오기

오늘은 새 도구가 하나 나옵니다. **`scipy.stats`** 입니다.
통계 계산을 해 주는 파이썬 꾸러미이고, Colab 에는 이미 설치돼 있습니다.

In [ ]:
import pandas as pd
from scipy import stats

BASE = 'https://raw.githubusercontent.com/JasonWhiteLee/ak-data-analysis-basics/main/'

orders  = pd.read_csv(BASE + 'superstore_orders.csv',
                      parse_dates=['Order Date', 'Ship Date']).drop_duplicates()
returns = pd.read_csv(BASE + 'superstore_returns.csv')

print(orders.shape)

## 그래프에 한글이 나오게 하기

4교시와 같습니다. 뒤에서 산점도를 그리므로 지금 실행해 둡니다.

In [ ]:
!apt-get -qq install fonts-nanum > /dev/null

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)

plt.plot([1, 2, 3], [1, 4, 2])
plt.title('한글이 보이면 성공입니다')
plt.show()

---
# 6-2. 검정이란 무엇인가

## 두 집단을 재면 언제나 숫자가 다르게 나옵니다

이건 아주 중요한 이야기입니다.

우리 회사 A팀 평균 근속연수와 B팀 평균 근속연수를 재면, **절대로 똑같이 나오지 않습니다.**
소수점 아래까지 재면 항상 다릅니다. 두 집단이 실제로는 아무 차이가 없어도 그렇습니다.

왜냐하면 우리가 재는 건 **몇 명의 실제 사람들**이고, 그 사람들이 누구냐에 따라 숫자가 흔들리기 때문입니다.

**그래서 "차이가 있느냐"는 질문은 의미가 없습니다.** 차이는 항상 있습니다.
진짜 질문은 이것입니다.

> **이 정도 차이가, 두 집단이 사실은 같은데도 그냥 흔들림으로 나올 만한 크기인가?**

## 동전으로 생각해 봅시다

멀쩡한 동전을 10번 던졌더니 앞면이 6번 나왔습니다.
**"이 동전 이상한데?"** 라고 하시겠습니까?

아마 아닐 겁니다. 멀쩡한 동전도 10번 중 6번쯤은 흔히 나옵니다.

그런데 **100번 던져서 60번**이 나왔다면 어떨까요? 슬슬 의심이 갑니다.
**1,000번 던져서 600번**이면 거의 확실히 이상한 동전입니다.

**앞면 비율은 셋 다 60%로 똑같습니다.** 그런데 우리 판단은 달라집니다.
왜냐하면 던진 횟수가 많아질수록 **"그냥 우연히 그렇게 나올 가능성"이 줄어들기 때문**입니다.

## 검정이 해 주는 일

통계 검정이 하는 일이 정확히 이겁니다.

1. **"두 집단은 사실 똑같다"고 일단 쳐 봅니다.** (이걸 귀무가설이라고 부릅니다)
2. 그렇게 쳤을 때, **내가 실제로 본 정도의 차이가 우연히 나올 가능성**을 계산합니다.
3. 그 가능성이 아주 낮으면 → "똑같다고 치기엔 무리다"라고 판단합니다.

계산은 컴퓨터가 합니다. **우리가 할 일은 "무엇과 무엇을 비교할지" 정하고, 나온 숫자를 읽는 것뿐입니다.**

---
# 6-3. p값 읽는 법

검정을 돌리면 **p값(p-value)** 이라는 숫자가 나옵니다. 0과 1 사이의 값입니다.

## p값의 뜻

> **두 집단이 사실 같다고 쳤을 때, 지금 본 정도의 차이가 우연히 나올 확률**

p값이 0.03 이면 — 두 집단이 같은데도 이런 차이가 나올 일이 100번 중 3번쯤 된다는 뜻입니다.
p값이 0.4 면 — 두 집단이 같아도 이런 차이는 100번 중 40번쯤 흔히 나온다는 뜻입니다.

## p값이 **아닌** 것 — 이게 더 중요합니다

| 흔한 오해 | 실제 |
|---|---|
| "p=0.03 이면 내 주장이 맞을 확률이 97%다" | **아닙니다.** p값은 내 주장의 확률이 아닙니다 |
| "p=0.4 면 두 집단이 같다는 뜻이다" | **아닙니다.** "다르다고 말할 근거가 부족하다"일 뿐입니다 |
| "p가 작으면 차이가 크다는 뜻이다" | **아닙니다.** 차이 크기와 p값은 다른 이야기입니다 (6-6에서 다룹니다) |

p값은 **"내 가설이 맞을 확률"이 아닙니다.**
**"두 집단이 같다고 가정했을 때 이런 결과가 나올 확률"**입니다. 방향이 반대입니다.

## 0.05 는 어디서 나온 숫자인가

보통 **p < 0.05 이면 "유의하다(significant)"** 고 말합니다.

**그런데 이 0.05 에는 아무런 수학적 근거가 없습니다.**
1920년대에 한 통계학자가 "20번에 1번 정도면 적당하지 않겠나" 하고 정한 관습입니다.
지금까지 그렇게 쓰고 있을 뿐입니다.

- p = 0.049 → 유의함
- p = 0.051 → 유의하지 않음

**이 둘 사이에 실질적인 차이는 없습니다.** 0.05 는 선을 그어 놓은 것이지 자연법칙이 아닙니다.

> **p값은 신호등이 아니라 온도계에 가깝습니다.**
> 초록/빨강으로 딱 갈리는 게 아니라, 숫자가 작을수록 "우연으로 보기 어렵다"가 점점 강해지는 것입니다.

---
# 6-4. 실제로 재 보기 (1) — 차이가 큰 경우

## 업무 질문을 분석 가설로 바꾸기

실무에서 오는 질문은 보통 이렇게 생겼습니다.

> **"할인 정책 계속 가도 됩니까?"**

이건 그대로는 분석할 수 없습니다. 컴퓨터가 계산할 수 있는 형태로 바꿔야 합니다.

| 단계 | 내용 |
|---|---|
| 업무 질문 | 할인 정책 계속 가도 됩니까? |
| ↓ 무엇을 비교할지 정한다 | **할인한 주문** vs **할인 안 한 주문** |
| ↓ 무엇을 잴지 정한다 | 주문당 **이익(Profit)** 의 평균 |
| ↓ 분석 가설 | "두 집단의 평균 이익은 같다" — 이걸 부술 수 있는지 본다 |

**이 표를 만드는 과정이 분석의 절반입니다.** 코드는 그다음입니다.

## 먼저 평균부터 봅니다

In [ ]:
할인함   = orders[orders['Discount'] > 0]['Profit']
할인안함 = orders[orders['Discount'] == 0]['Profit']

print('할인한 주문   : {:>5}건   평균 이익 {:8.2f}'.format(len(할인함), 할인함.mean()))
print('할인 안 한 주문: {:>5}건   평균 이익 {:8.2f}'.format(len(할인안함), 할인안함.mean()))

**할인한 주문은 평균 -6.53, 할인 안 한 주문은 평균 66.34.**

할인한 주문은 **평균적으로 적자**입니다. 눈으로 봐도 차이가 커 보입니다.
그런데 정말 그런지 재 봅시다.

## t-test — 두 집단의 평균을 비교하는 도구

**두 집단**의 **평균**을 비교할 때 쓰는 것이 **t-test** 입니다.

```
stats.ttest_ind(집단A, 집단B, equal_var=False)
```

- `ttest_ind` — independent(독립된) 두 집단의 t-test 라는 뜻입니다
- `equal_var=False` — **"두 집단의 흩어짐 정도가 같다고 가정하지 마라"** 는 뜻입니다.
  실무에서는 두 집단의 흩어짐이 같을 이유가 없으므로 **이걸 붙이는 게 안전합니다.**
  헷갈리면 그냥 항상 `equal_var=False` 를 쓰세요.

In [ ]:
결과 = stats.ttest_ind(할인함, 할인안함, ____)

print('평균 차이: {:.2f}'.format(할인함.mean() - 할인안함.mean()))
print('p값      :', 결과.pvalue)

## p값이 0.000000000000000000000000000000000000000000000000000000037 입니다

`3.727e-56` 이라고 나왔을 겁니다. **e-56 은 "소수점 아래 56자리"** 라는 뜻입니다.
0.0000... 으로 쓰면 화면을 다 채우니까 이렇게 줄여 씁니다.

**읽는 법:** 두 집단이 사실 같은데도 이 정도 차이가 우연히 나올 확률이 사실상 0이라는 뜻입니다.

**해석:** 할인한 주문과 안 한 주문의 이익 차이는 **우연으로 보기 어렵습니다.**

## 하나 더 — 가구 vs 기술

In [ ]:
가구 = orders[orders['Category'] == 'Furniture']['Profit']
기술 = orders[orders['Category'] == 'Technology']['Profit']

결과 = stats.ttest_ind(가구, 기술, equal_var=False)

print('가구 평균 이익: {:7.2f}  ({}건)'.format(가구.mean(), len(가구)))
print('기술 평균 이익: {:7.2f}  ({}건)'.format(기술.mean(), len(기술)))
print('p값:', 결과.pvalue)

**가구 8.96 / 기술 78.58, p값은 0.000000000017 (1.7e-11).**

기술 제품이 가구보다 훨씬 남습니다. 이것도 우연으로 보기 어렵습니다.

> ### 여기까지는 쉬웠습니다
> 눈으로 봐도 차이가 컸고, 검정도 그렇다고 했습니다.
> **문제는 다음 절입니다.**

---
# 6-5. 실제로 재 보기 (2) — 차이가 **없는** 경우

## 4교시 숙제를 회수합니다

4교시에서 고객 유형별 주문 금액을 상자그림으로 그렸습니다.
상자 세 개가 거의 같은 높이에 있었고, 이렇게 적어 뒀습니다.

> *"이게 '차이가 없다'는 뜻일까요, 아니면 '눈으로는 모르겠다'는 뜻일까요?"*

**이제 재 봅니다.** 이번에는 이익(Profit)으로 봅니다.

In [ ]:
개인   = orders[orders['Segment'] == 'Consumer']['Profit']
법인   = orders[orders['Segment'] == 'Corporate']['Profit']

print('Consumer  평균 이익: {:6.2f}   ({}건)'.format(개인.mean(), len(개인)))
print('Corporate 평균 이익: {:6.2f}   ({}건)'.format(법인.mean(), len(법인)))
print('차이: {:.2f}'.format(법인.mean() - 개인.mean()))

## 눈으로 보면 이렇습니다

- Consumer 25.82
- Corporate 30.50
- **차이 4.68**

주문 한 건당 4.68 씩 법인이 더 남습니다. 5,000건이면 2만이 넘습니다.

**"법인 영업에 힘을 더 실어야겠는데요."** — 회의에서 이런 말이 나올 만한 숫자입니다.

**재 봅시다.**

In [ ]:
결과 = stats.ttest_ind(개인, 법인, equal_var=False)

print('p값: {:.4f}'.format(결과.____))

## p = 0.3781

**두 집단이 사실 같아도, 이 정도 차이는 100번 중 38번쯤 그냥 나옵니다.**

즉 **4.68 이라는 차이는 그냥 흔들림 범위 안에 있습니다.**
이 데이터만 가지고 "법인이 더 남는다"고 말할 근거가 없습니다.

## 조심할 것 — "차이가 없다"고 말하면 안 됩니다

p값이 크게 나왔을 때 할 수 있는 말은 이겁니다.

| 이렇게 말하면 안 됩니다 | 이렇게 말합니다 |
|---|---|
| "두 집단은 같다" | "**다르다고 말할 근거를 찾지 못했다**" |
| "차이가 없다는 게 증명됐다" | "**이 데이터로는 판단할 수 없다**" |

차이가 진짜 없을 수도 있고, 차이가 있는데 **데이터가 부족해서 못 잡은 것**일 수도 있습니다.
검정은 이 둘을 구분해 주지 못합니다.

## 하나 더 — 반품과 주문 금액

3교시에서 반품 데이터를 붙였습니다. **"금액이 큰 주문이 더 많이 반품될까?"** 를 봅시다.

In [ ]:
합친표 = orders.merge(returns, on='Order ID', how='left')
합친표['Returned'] = 합친표['Returned'].fillna('No')

반품됨   = 합친표[합친표['Returned'] == 'Yes']['Sales']
반품안됨 = 합친표[합친표['Returned'] == 'No']['Sales']

결과 = stats.ttest_ind(반품됨, 반품안됨, equal_var=False)

print('반품된 주문   평균 금액: {:7.2f}  ({}건)'.format(반품됨.mean(), len(반품됨)))
print('반품 안 된 주문 평균 금액: {:7.2f}  ({}건)'.format(반품안됨.mean(), len(반품안됨)))
print('p값: {:.4f}'.format(결과.pvalue))

**233.39 vs 239.70, p = 0.8121.**

거의 100번 중 81번은 그냥 나올 차이입니다.
**"비싼 물건이 더 반품된다"는 이야기는 이 데이터에서 나오지 않습니다.**

> ### 이 절이 오늘의 핵심입니다
> **차이는 항상 있습니다. 문제는 그게 의미 있는 차이인가입니다.**
>
> 표를 뽑아서 "Corporate 이 4.68 더 높습니다"라고 보고하면 그건 틀린 말은 아닙니다.
> 하지만 **그 숫자를 근거로 뭔가를 결정하면 안 됩니다.** 그냥 흔들림이니까요.
>
> 검정을 돌리는 이유가 이겁니다. **엑셀 피벗으로 나온 차이를 그대로 믿지 않기 위해서.**

---
# 6-6. 유의하다 ≠ 중요하다

이번엔 반대 방향의 함정입니다.

## 표본이 크면 아주 작은 차이도 p값이 작게 나옵니다

앞에서 동전 이야기를 했습니다. 60%가 나와도 10번 던진 거면 우연이지만, 1,000번 던진 거면 확실했죠.
**같은 원리가 반대로도 작동합니다.**

데이터가 아주 많으면, **아무리 작은 차이도** "우연으로 보기 어렵다"는 결론이 나옵니다.

직접 확인해 봅시다. 이익이 딱 0.5 만큼만 차이 나는 가짜 데이터를 만들어 보겠습니다.

In [ ]:
import numpy as np
np.random.seed(0)

for 크기 in [100, 1000, 100000]:
    A = np.random.normal(100,     10, 크기)   # 평균 100
    B = np.random.normal(100.5,   10, 크기)   # 평균 100.5 — 딱 0.5 차이
    p = stats.ttest_ind(A, B, equal_var=False).pvalue
    print('표본 {:>6}개씩 → 실제 평균차 0.5 → p값 {:.4f}'.format(크기, p))

## 같은 0.5 차이인데 p값이 완전히 다릅니다

100개씩일 때는 "우연"이고, 100,000개씩이면 "확실한 차이"가 됩니다.
**차이의 크기는 셋 다 똑같이 0.5 인데도요.**

> ### 그래서 이렇게 됩니다
> 요즘 회사 데이터는 수십만 건입니다. **웬만한 차이는 다 유의하게 나옵니다.**
> "p < 0.05 니까 의미 있는 차이입니다"라는 보고는 **거의 아무 정보도 담고 있지 않습니다.**

## 반드시 두 가지를 함께 보세요

| 보는 것 | 답하는 질문 |
|---|---|
| **p값** | 이 차이가 우연일 만한가? |
| **차이의 크기** (평균 차, 비율 차) | 그래서 **업무적으로 신경 쓸 만한 크기인가?** |

앞의 할인 사례로 돌아가면 —

- p값: 3.7e-56 → 우연이 아니다
- **평균 차: -72.87** → 할인한 주문이 건당 73 씩 덜 남는다

**후자가 회의에서 실제로 쓰이는 숫자입니다.** p값은 "이 73이 헛것이 아니다"를 보증해 줄 뿐입니다.

**보고할 때는 항상 이 순서로 쓰세요.**
> "할인한 주문은 건당 평균 이익이 **73 낮습니다.** (p < 0.001)"

숫자가 앞이고 p값은 괄호 안입니다.

---
# 6-7. West 반품률 다시 보기 — 3교시 숙제 회수

3교시에서 지역별 반품률을 냈고, 이렇게 적어 뒀습니다.

> *"이 차이가 의미 있는 차이인지 판단하는 방법은 6교시에 다룹니다."*

**이제 다룹니다.** 먼저 그 표를 다시 만듭니다.

In [ ]:
주문단위 = 합친표.groupby('Order ID').agg(
    Region=('Region', 'first'),
    반품=('Returned', 'first'),
)
주문단위['반품'] = (주문단위['반품'] == 'Yes')

print('주문 건수:', len(주문단위))
(주문단위.groupby('Region')['반품'].mean() * 100).round(2)

| 지역 | 반품률 |
|---|---|
| Central | 3.31% |
| East | 2.98% |
| South | 2.92% |
| **West** | **11.56%** |

West 만 **세 배 넘게** 높습니다.

## 그런데 여기엔 t-test 를 쓸 수 없습니다

왜 그럴까요?

**t-test 는 "평균"을 비교하는 도구**입니다. 키, 금액, 이익 같은 **숫자**의 평균이요.
그런데 지금 우리가 비교하는 건 **반품됐다 / 안 됐다**라는 **범주**입니다. 평균을 낼 대상이 아닙니다.

게다가 비교할 집단이 **넷**입니다. t-test 는 **둘**을 비교하는 도구입니다.

**범주와 범주 사이에 관련이 있는지 볼 때 쓰는 것이 카이제곱(chi-square) 검정입니다.**

| 지금 상황 | |
|---|---|
| 범주 1 | 지역 (Central / East / South / West) |
| 범주 2 | 반품 여부 (Yes / No) |
| 질문 | **이 두 가지가 서로 관련이 있는가?** |

## 카이제곱이 하는 일 — 말로 설명하면

1. **"지역과 반품은 아무 관련이 없다"고 일단 쳐 봅니다.**
2. 그렇다면 각 지역의 반품 건수가 **몇 건쯤이어야 하는지** 계산합니다. (전체 반품률을 지역 크기에 맞춰 나눈 값입니다)
3. **그 예상치와 실제 숫자가 얼마나 어긋나는지** 잽니다.
4. 어긋남이 우연으로 보기 어려울 만큼 크면 → "관련이 있다"

## 먼저 표를 만듭니다 — `pd.crosstab`

카이제곱에는 **비율이 아니라 실제 건수 표**가 필요합니다.
`pd.crosstab` 은 두 범주를 가로세로로 놓고 건수를 세 주는 함수입니다. (엑셀 피벗의 개수 세기와 같습니다)

In [ ]:
교차표 = pd.____(주문단위['Region'], 주문단위['반품'])
교차표

`False` 열이 반품 안 된 주문, `True` 열이 반품된 주문입니다.

이제 이 표를 그대로 카이제곱에 넣습니다.

In [ ]:
카이제곱값, p값, 자유도, 기대값 = stats.____(교차표)

print('카이제곱값: {:.1f}'.format(카이제곱값))
print('p값       :', p값)

## 카이제곱값 146.8, p값 0.00000000000000000000000000000013 (1.3e-31)

**우연으로 보기 어렵습니다.** 지역과 반품 여부는 관련이 있습니다.

**"West 만 유난히 높은 게 그냥 어쩌다 그런 것"이라는 설명은 이제 못 씁니다.**

## 어긋남을 눈으로 봅시다

`stats.chi2_contingency` 가 돌려준 네 번째 값 `기대값` 이 바로
**"지역과 반품이 아무 관련 없다면 이랬을 건수"** 입니다.

In [ ]:
기대 = pd.DataFrame(기대값, index=교차표.index, columns=교차표.columns).round(1)

비교 = pd.DataFrame({
    '실제_반품': 교차표[True],
    '예상_반품': 기대[True],
})
비교['차이'] = (비교['실제_반품'] - 비교['예상_반품']).round(1)
비교

## 표를 읽어 보세요

| 지역 | 실제 반품 | 관련 없었다면 | 차이 |
|---|---|---|---|
| Central | 39 | 68.3 | **-29.3** |
| East | 44 | 85.4 | **-41.4** |
| South | 24 | 47.6 | **-23.6** |
| **West** | **189** | **94.7** | **+94.3** |

West 는 **예상의 두 배**가 반품됐습니다. 나머지 세 지역은 모두 예상보다 적습니다.

**차이는 전부 West 에서 나오고 있습니다.**

> ### 그래서 결론이 뭡니까
> **"West 반품률이 높은 것은 우연이 아니다"** — 여기까지가 통계가 말해 주는 전부입니다.
>
> **왜 높은지는 통계가 말해 주지 않습니다.** 3교시에서 적었던 그 질문들이 그대로 남아 있습니다.
>
> - West 가 실제로 반품이 많은가?
> - West 만 반품 기록을 꼼꼼히 남기는가? (기록 관행의 차이)
> - West 가 반품 많은 품목을 주로 파는가? (품목 구성의 차이)
>
> **통계는 "이건 우연이 아니니 진짜 알아볼 가치가 있다"까지만 말해 줍니다.**
> 그다음은 West 지점에 전화를 걸어서 알아내야 합니다. 이게 분석가가 하는 일입니다.

---
# 6-8. 상관 — 두 숫자가 같이 움직이는가

여기까지는 **집단을 나눠서** 비교했습니다.
이번에는 **숫자 두 개가 함께 움직이는지**를 봅니다.

- 할인율이 올라가면 이익이 내려가는가?
- 주문 금액이 크면 이익도 큰가?

이걸 재는 숫자가 **상관계수**입니다. `df.corr()` 한 줄이면 나옵니다.

In [ ]:
orders[['Sales', 'Quantity', 'Discount', 'Profit']].____.round(3)

## 상관계수 읽는 법

상관계수는 **-1 에서 +1 사이**의 값입니다.

| 값 | 뜻 |
|---|---|
| **+1 에 가까움** | 하나가 오르면 다른 하나도 오른다 |
| **0 에 가까움** | 같이 움직이는 관계가 잘 안 보인다 |
| **-1 에 가까움** | 하나가 오르면 다른 하나는 내려간다 |

대각선이 전부 1.000 인 건, 자기 자신과의 상관이라 당연한 값입니다. 무시하세요.

**우리 데이터에서 눈여겨볼 값**

| 짝 | 상관계수 |
|---|---|
| Sales ↔ Profit | **+0.243** |
| Discount ↔ Profit | **-0.219** |
| Quantity ↔ Profit | +0.065 |

Discount 와 Profit 이 **음수**입니다. 할인이 올라가면 이익이 내려가는 방향입니다. 예상대로입니다.

**그런데 -0.219 는 얼마나 강한 걸까요?**

## "약한 상관"이 무슨 뜻인지 눈으로 봅시다

거칠게 이렇게들 부릅니다. (이것도 관습일 뿐 기준선은 아닙니다)

| 절댓값 | 흔히 부르는 말 |
|---|---|
| 0.0 ~ 0.2 | 거의 없음 |
| 0.2 ~ 0.4 | **약함** |
| 0.4 ~ 0.7 | 중간 |
| 0.7 ~ | 강함 |

-0.219 는 **약한 축**입니다. 말로만 들으면 감이 안 오니 그림을 그려 봅시다.

In [ ]:
표시용 = orders[orders['Profit'].between(-500, 500)]

표시용.plot(kind='scatter', x='Discount', y='Profit',
            alpha=0.15, figsize=(9, 5))
plt.axhline(0, color='red', linewidth=1)
plt.title('할인율과 이익 (상관 -0.219)')
plt.xlabel('할인율')
plt.ylabel('이익')
plt.show()

## 이게 상관 -0.219 의 실제 모습입니다

**깔끔한 선이 아닙니다.** 점이 사방에 흩어져 있습니다.

자세히 보면 이런 게 보입니다.
- 할인 0 인 쪽은 점이 대부분 빨간 선(이익 0) **위**에 있습니다
- 할인이 커질수록 빨간 선 **아래**로 내려간 점이 많아집니다

**경향은 있습니다. 하지만 예측은 못 합니다.**

> ### 상관계수를 볼 때 꼭 기억할 것
> **"할인율 30%인 주문이 있는데, 이익이 얼마일까요?"**
> 이 그림으로는 답할 수 없습니다. 30% 자리에 위아래로 점이 잔뜩 깔려 있으니까요.
>
> 상관 -0.219 가 말해 주는 것은 **"전체적으로 그런 경향이 있다"** 까지입니다.
> **개별 주문 하나를 맞히지는 못합니다.**
>
> 그래서 상관계수는 **숫자만 보지 말고 반드시 그림을 함께 그려 보세요.**
> 같은 상관계수라도 그림 모양은 아주 다를 수 있습니다.

## 하나 더 — 상관은 "직선 관계"만 봅니다

상관계수는 **"한쪽이 오르면 다른 쪽도 일정하게 오르는가"**만 잽니다.

그래서 예를 들어 **"광고비를 어느 정도까지 늘리면 매출이 오르다가, 그 이상은 효과가 없다"**
같은 관계는 상관계수가 낮게 나옵니다. 관계가 없어서가 아니라 **직선이 아니라서**입니다.

**상관계수가 0에 가깝다고 "관계없음"이라고 결론 내리지 마세요.** 그림을 보세요.

---
# 6-9. 상관은 인과가 아니다

**오늘 가장 중요한 절입니다.**

지금까지 배운 걸로 이런 보고서를 쓸 수 있습니다.
"A와 B는 상관이 있고, 검정 결과 우연이 아닙니다."

**그다음에 거의 항상 이런 말이 따라옵니다.** — "그럼 A를 바꾸면 B가 바뀌겠네요?"

**아닙니다.** 왜 아닌지 실제 데이터로 보겠습니다.

## 새 데이터 — 브라질 온라인 커머스 (Olist)

실제 브라질 전자상거래 회사의 주문 10만 건과 고객 리뷰 데이터입니다.
파일이 커서 불러오는 데 시간이 좀 걸립니다.

In [ ]:
oo = pd.read_csv(BASE + 'olist/olist_orders_dataset.csv',
                 parse_dates=['order_purchase_timestamp',
                              'order_delivered_customer_date',
                              'order_estimated_delivery_date'])
rv = pd.read_csv(BASE + 'olist/olist_order_reviews_dataset.csv')

print('주문:', oo.shape)
print('리뷰:', rv.shape)

In [ ]:
배송 = oo.merge(rv[['order_id', 'review_score']], on='order_id')

# 실제 도착일 - 약속한 도착일 (양수면 늦은 것)
배송['지연일수'] = (배송['order_delivered_customer_date']
                 - 배송['order_estimated_delivery_date']).dt.days

배송 = 배송.dropna(subset=['지연일수', 'review_score'])
배송['약속어김'] = 배송['지연일수'] > 0

배송.groupby('약속어김')['review_score'].agg(['mean', 'count']).round(2)

## 결과

| | 평균 별점 | 건수 |
|---|---|---|
| **약속한 날짜를 지킴** | **4.29** | 89,949 |
| **약속한 날짜를 넘김** | **2.27** | 6,410 |

**별점이 2점 넘게 차이납니다.** 5점 만점에서 2점 차이는 엄청난 차이입니다.

재 봅시다.

In [ ]:
늦음 = 배송[배송['약속어김']]['review_score']
정시 = 배송[~배송['약속어김']]['review_score']

결과 = stats.____(늦음, 정시, equal_var=False)

print('평균 차이: {:.2f}점'.format(늦음.mean() - 정시.mean()))
print('p값      :', 결과.pvalue)
print()
print('지연일수와 별점의 상관: {:.3f}'.format(배송['지연일수'].corr(배송['review_score'])))

**p값이 그냥 `0.0` 으로 나왔습니다.**

이건 진짜 0이 아니라, **컴퓨터가 표시할 수 있는 가장 작은 수보다 작다**는 뜻입니다.

- 평균 차이: **-2.02점**
- 상관계수: **-0.267** (늦을수록 별점이 낮다)
- p값: 표시 불가능할 만큼 작음

**차이의 크기도 크고(2점), p값도 확실합니다.** 6-6에서 배운 두 가지가 다 만족됐습니다.

---

# 그럼 이렇게 보고해도 됩니까?

> ### **"배송을 빨리 하면 별점이 오릅니다. 물류에 투자합시다."**

## 답: **알 수 없습니다.**

데이터가 말해 준 것은 **"늦은 주문의 별점이 낮다"** 입니다.
**"늦게 배송했기 때문에 별점이 낮아졌다"** 는 말이 아닙니다.

## 왜 아닐까요 — 늦게 배송된 주문이 어떤 주문인지 생각해 봅시다

배송이 늦은 주문은 그냥 아무 주문이나가 아닙니다. **애초에 다른 주문들입니다.**

| 늦은 이유 | 별점이 낮은 다른 이유 |
|---|---|
| **먼 지역**이라 오래 걸렸다 | 먼 지역은 운송 중 파손도 많고, 반품도 어렵다 |
| **재고가 없어서** 늦게 발송됐다 | 재고 관리가 부실한 판매자의 상품이다 |
| **무겁거나 큰 물건**이라 늦었다 | 조립이 필요하거나 설치가 번거로운 상품이다 |
| **영세 판매자**라 처리가 늦었다 | 포장·응대·상품 설명도 부실했다 |

**이 중 어느 것이든 별점을 낮출 수 있습니다.** 배송 속도와 상관없이요.

즉 **"늦음"과 "낮은 별점"이 함께 나타났지만, 둘 다 세 번째 원인 때문일 수 있습니다.**

## 인과를 말하려면 무엇이 필요한가

인과관계를 주장한다는 것은 이 질문에 답한다는 뜻입니다.

> ### **"그 주문들을 만약 제때 배송했다면, 별점이 올랐을까?"**

**그런데 그 주문들은 이미 늦게 배송됐습니다.** 되돌릴 수 없습니다.
"만약 그러지 않았다면"의 세계는 우리가 관찰할 수 없습니다. 이걸 **반사실**이라고 부릅니다.

우리 데이터에는 이 답이 **없습니다.** 앞으로도 없습니다. 데이터를 더 모아도 없습니다.

## 그래서 A/B 테스트를 합니다

관찰만으로 답할 수 없다면, **답할 수 있는 상황을 직접 만드는 수밖에 없습니다.**

1. 들어오는 주문을 **동전 던지듯 무작위로** 두 무리로 나눕니다
2. 한 무리는 **빠른 배송**, 다른 무리는 평소대로
3. 별점을 비교합니다

**무작위로 나눴기 때문에**, 두 무리에는 먼 지역도 가까운 지역도, 무거운 물건도 가벼운 물건도
**골고루 섞여 있습니다.** 두 무리의 유일한 차이는 우리가 바꾼 것 하나뿐입니다.

**그래서 여기서 나온 차이는 인과라고 말할 수 있습니다.**

---

> ### 보고서에 쓸 때 — 단어를 조심하세요
>
> | 이런 표현 | 뜻 | 관찰 데이터로 |
> |---|---|---|
> | "늦은 주문은 별점이 낮습니다" | 함께 나타난다 | **가능** |
> | "늦으면 별점이 낮아집니다" | 원인이다 | **불가능** |
> | "배송을 개선하면 별점이 오릅니다" | 원인이고 개입 효과까지 | **불가능** |
>
> **첫 줄은 데이터가 보증합니다. 아래 두 줄은 여러분이 덧붙인 것입니다.**
>
> 회의에서 아래 두 줄을 말해야 할 때는, **"이건 제 해석입니다"** 라고 붙이세요.
> 그 한마디가 나중에 여러분을 지켜 줍니다.

## 그렇다고 아무것도 하지 말라는 뜻이 아닙니다

"인과가 아니니까 아무 말도 하면 안 된다"는 게 아닙니다.

**배송 지연이 별점 하락과 강하게 함께 나타난다** — 이건 **알아볼 가치가 충분한 신호**입니다.
할 일은 이것입니다.

1. 상관을 발견한다 (지금 한 것)
2. **그럴듯한 다른 설명들을 목록으로 적는다** (먼 지역? 무거운 물건? 영세 판매자?)
3. 가능하면 그것들을 **나눠서 확인한다** (예: 같은 지역 안에서만 비교해 본다)
4. 그래도 남으면 **A/B 테스트로 확인한다**

**2번을 건너뛰는 것이 실무에서 가장 자주 나오는 실수입니다.**

---
# 6-10. 회귀는 언제 쓰나

바로 앞에서 이런 문제를 만났습니다.

> "배송이 늦어서 별점이 낮은 건가, 아니면 **먼 지역이라서** 낮은 건가?"

이 둘을 **동시에 놓고** 각각의 몫을 나눠 보고 싶습니다. 그럴 때 쓰는 것이 **회귀분석**입니다.

## 한 줄 설명

> **여러 요인이 결과에 동시에 영향을 줄 때, 각 요인의 몫을 나눠서 보여 주는 방법**

상관계수는 **두 개씩만** 볼 수 있습니다. 회귀는 **여러 개를 한꺼번에** 넣습니다.

## 결과가 어떻게 나오는가

별점을 배송지연·거리·상품가격으로 회귀분석하면 대략 이런 식으로 나옵니다.

| 요인 | 몫 |
|---|---|
| 배송 지연 (하루당) | -0.05점 |
| 배송 거리 (100km당) | -0.01점 |
| 상품 가격 (만원당) | +0.02점 |

**읽는 법:** "거리와 가격이 같은 주문들끼리 비교했을 때, 지연이 하루 늘 때마다 별점이 0.05 낮다"

**"다른 조건이 같다면"** — 이게 회귀가 해 주는 일입니다.

## 언제 쓰는가

| 상황 | |
|---|---|
| **여러 요인 중 어느 게 더 중요한지** 알고 싶다 | ✔ |
| **다른 조건을 맞춘 상태에서** 비교하고 싶다 | ✔ |
| 값을 **예측**하고 싶다 (내년 매출, 이탈 가능성) | ✔ |

## 오늘은 여기까지만 합니다

회귀는 제대로 하려면 확인할 것이 꽤 많습니다.
어떤 요인을 넣을지, 요인끼리 겹치지 않는지, 관계가 직선인지 등등.

**오늘 기억할 것은 이것 하나입니다.**

> **"요인이 여러 개 얽혀 있어서 각각의 몫을 나눠 보고 싶다"** 고 느낄 때가 오면,
> 그때 찾아볼 것의 이름이 **회귀분석**입니다.

> ### 여전히 인과는 아닙니다
> 회귀에 요인을 아무리 많이 넣어도 **넣지 않은(혹은 모르는) 요인**은 남아 있습니다.
> 회귀는 "다른 조건이 같다면"을 **우리가 넣은 것들에 한해서만** 맞춰 줍니다.
> **6-9의 이야기는 회귀를 써도 그대로 유효합니다.**

---
# 6-11. 어떤 방법을 언제 쓰나 — 표 한 장

오늘 나온 것들을 한 장으로 모았습니다. **이 표만 챙겨 가셔도 됩니다.**

| 하고 싶은 것 | 방법 | 코드 |
|---|---|---|
| **두 집단의 평균** 비교 | t-test | `stats.ttest_ind(A, B, equal_var=False)` |
| **세 집단 이상의 평균** 비교 | ANOVA (분산분석) | `stats.f_oneway(A, B, C)` |
| **범주와 범주의 관련성** | 카이제곱 | `stats.chi2_contingency(pd.crosstab(a, b))` |
| **숫자 두 개가 같이 움직이는가** | 상관 | `df.corr()` |
| **여러 요인의 몫을 나눠 보기** | 회귀 | (오늘은 개념만) |

## 고르는 순서 — 두 가지만 물어보세요

**질문 1. 내가 비교하려는 것이 숫자입니까, 범주입니까?**

- 금액·개수·점수·일수 → **숫자**
- 지역·반품여부·고객유형·예/아니오 → **범주**

**질문 2. 집단으로 나눠서 비교합니까, 아니면 두 숫자의 동행을 봅니까?**

| | 나눠서 비교 | 동행 |
|---|---|---|
| **숫자** | t-test (2개) / ANOVA (3개 이상) | 상관 |
| **범주** | 카이제곱 | 카이제곱 |

## 세 집단 이상일 때 — 왜 t-test 를 여러 번 하면 안 되나

지역 넷을 비교하고 싶어서 t-test 를 6번 (Central-East, Central-South, ...) 돌리면 어떻게 될까요?

**p < 0.05 는 "우연히 유의하게 나올 확률이 20번에 1번"이라는 뜻입니다.**
6번 돌리면 그중 하나쯤은 그냥 우연히 유의하게 나옵니다.

**여러 번 재면 우연히 걸리는 게 생깁니다.** 그래서 셋 이상은 ANOVA 로 한 번에 봅니다.

> 이건 실무에서도 그대로 적용됩니다.
> **표를 스무 개 뽑아서 그중 눈에 띄는 하나를 골라 보고하면**, 그 하나가 우연일 가능성이 큽니다.
> 무엇을 볼지는 **데이터를 보기 전에** 정하는 게 좋습니다. 6-1에서 예상을 먼저 적게 한 이유이기도 합니다.

---
# 6-12. 예상과 대조하기

**6-1로 올라가서 여러분이 적었던 예상을 다시 읽어 보세요.**

그리고 아래 셀에 결과와 대조해서 적어 주세요.

---

### 아래 셀을 더블클릭해서 채우세요

| # | 질문 | 내 예상 | 실제 결과 | 맞았나 |
|---|---|---|---|---|
| 1 | 할인 유무에 따른 이익 차이 | | -6.53 vs 66.34 (p=3.7e-56) 차이 확실 | |
| 2 | 고객 유형에 따른 이익 차이 | | 25.82 vs 30.50 (p=0.38) **근거 없음** | |
| 3 | West 반품률이 진짜 차이인가 | | 카이제곱 p=1.3e-31 **우연 아님** | |
| 4 | 할인율과 이익의 연결 강도 | | 상관 -0.219 **약함** | |
| 5 | 배송을 빨리 하면 별점이 오르나 | | **알 수 없음** (관찰 데이터로는 불가) | |

**빗나간 것이 있었다면, 어느 것이었고 왜 그렇게 예상했는지 적어 주세요.**

　→

**오늘 배운 것 중 내 업무에서 당장 다시 봐야 할 숫자가 있습니까?**
(예: "우리 팀이 매달 보고하는 그 차이, 검정해 본 적이 없다")

　→

---

> ### 빗나간 사람이 오늘 가장 많이 배운 사람입니다
>
> 다섯 개를 다 맞히셨다면, 오늘 새로 안 것은 **파이썬 명령어 몇 개**뿐입니다.
>
> 하나라도 빗나가셨다면, **여러분이 원래 갖고 있던 생각 하나가 데이터에 의해 고쳐진 것**입니다.
> 그게 분석을 하는 이유입니다.
>
> 특히 2번(고객 유형)에서 "Corporate 이 더 남을 것"이라고 적으셨다면 —
> 그건 틀린 예상이 아니라 **아주 자연스러운 예상**입니다. 숫자상으로 실제로 높았으니까요.
> 다만 그 숫자가 **결정의 근거가 되기엔 부족했을 뿐**입니다.
> **이걸 구분할 줄 아는 것이 오늘의 전부입니다.**

---
# 정리 — 오늘 쓴 것

## 코드

| 하는 일 | 코드 |
|---|---|
| 통계 도구 불러오기 | `from scipy import stats` |
| 두 집단 평균 비교 | `stats.ttest_ind(A, B, equal_var=False)` |
| p값만 꺼내기 | `결과.pvalue` |
| 세 집단 이상 평균 비교 | `stats.f_oneway(A, B, C)` |
| 교차표(건수) 만들기 | `pd.crosstab(df['범주1'], df['범주2'])` |
| 범주 간 관련성 검정 | `stats.chi2_contingency(교차표)` |
| 상관계수 표 | `df[['A','B','C']].corr()` |
| 두 열의 상관만 | `df['A'].corr(df['B'])` |
| 산점도 | `df.plot(kind='scatter', x='A', y='B', alpha=0.15)` |

## 남길 것 세 가지

1. **차이는 항상 있다. 문제는 의미 있는 차이인가다.**
   Consumer 25.82 vs Corporate 30.50 — 눈으로는 차이지만 p=0.38, 흔들림 범위였습니다.

2. **p값과 차이의 크기를 함께 본다.**
   데이터가 많으면 아무 차이나 유의해집니다. 보고할 때는 **차이 크기가 앞, p값은 괄호 안.**

3. **상관은 인과가 아니다.**
   "늦은 주문은 별점이 낮다"는 데이터가 보증하지만,
   "늦어서 별점이 낮아졌다"는 우리가 덧붙인 해석입니다. 확인하려면 A/B 테스트가 필요합니다.

## 오늘 회수한 숙제

| 어디서 | 무엇을 | 결론 |
|---|---|---|
| 3교시 | West 반품률 11.56% 가 진짜 차이인가 | 카이제곱 p=1.3e-31 — **우연 아님. 원인은 따로 조사할 일** |
| 4교시 | 상자그림에서 고객 유형별 차이가 없어 보였는데 | t-test p=0.38 — **다르다고 말할 근거 없음** |

---

### 다음 시간

**7교시: 종합 실습 — 그래서 무엇을 다르게 할 것인가**

오늘까지 배운 것을 처음부터 끝까지 한 번에 해 봅니다.
데이터를 불러오고, 정리하고, 살펴보고, 검정하고, 그림을 그리고 —
**마지막에 "그래서 무엇을 다르게 할 것인가"를 한 장으로 씁니다.**

분석의 결과물은 표나 그래프가 아니라 **바뀐 행동**입니다. 그걸 만들어 보는 시간입니다.